# SAE Feature Dashboard (no training)

This notebook builds dashboard artifacts for an already trained SAE: it loads a CSV dataset from `data/`, extracts token-level activations from a selected Qwen layer, encodes them with the SAE, computes concept-level and token-level attributions, and saves dashboard files back to `data/`.

This notebook **does not train SAE**. It expects an existing checkpoint compatible with the selected model and layer.

In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import json
import os
import sys
import warnings

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "examples" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from backend.sae_io import load_sae_checkpoint, resolve_project_path
from backend.schemas import DEFAULT_LAYER_NAME, DEFAULT_MODEL_NAME, DEFAULT_SAE_CHECKPOINT_PATH
from interpretability.sae.feature_analysis import (
    attribute_features_to_concepts,
    top_examples_for_feature,
    top_tokens_for_features,
)
from utils.inference_utils.llm import LLM

warnings.filterwarnings("default")
torch.set_grad_enabled(False)

## 1. Configuration

In [ ]:
DATASET_PATH = PROJECT_ROOT / "data" / "sae_experimental_dataset.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "sae_feature_dashboard"

MODEL_NAME_OR_PATH = DEFAULT_MODEL_NAME
SAE_CHECKPOINT_PATH = resolve_project_path(DEFAULT_SAE_CHECKPOINT_PATH)
LAYER_NAME = DEFAULT_LAYER_NAME
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_DATASET_ROWS: int | None = 16
MAX_LENGTH = 256
BATCH_SIZE = 2

TOP_K_FEATURES_PER_CONCEPT = 10
TOP_K_TOKENS_PER_FEATURE = 30
TOP_K_EXAMPLES_PER_FEATURE = 10
TOP_K_FEATURES_PER_TOKEN = 5
MAX_DASHBOARD_FEATURES = 80

SCORE_METHOD = "cohen_d"
TOKEN_AGGREGATION = "mean_abs"
ACTIVATION_THRESHOLD = 0.0
CONTEXT_WINDOW = 5

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

config = {
    "dataset_path": str(DATASET_PATH),
    "output_dir": str(OUTPUT_DIR),
    "model_name_or_path": MODEL_NAME_OR_PATH,
    "sae_checkpoint_path": str(SAE_CHECKPOINT_PATH),
    "layer_name": LAYER_NAME,
    "device": DEVICE,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "score_method": SCORE_METHOD,
    "token_aggregation": TOKEN_AGGREGATION,
}
config

## 2. Dataset loading
`text` and `concept_label` are strictly required

In [ ]:
dataset_df = pd.read_csv(DATASET_PATH, dtype=str).fillna("")
required_columns = {"text", "concept_label"}
missing_columns = required_columns - set(dataset_df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

if "sample_id" not in dataset_df.columns:
    dataset_df["sample_id"] = [f"sample_{idx:05d}" for idx in range(len(dataset_df))]

if MAX_DATASET_ROWS is not None:
    dataset_df = dataset_df.head(MAX_DATASET_ROWS).copy()

texts = dataset_df["text"].astype(str).tolist()
concept_labels = dataset_df["concept_label"].astype(str).tolist()
sample_ids = dataset_df["sample_id"].astype(str).tolist()

print(f"samples={len(dataset_df)}")
display(dataset_df[["sample_id", "concept_label", "text"]].head())
display(dataset_df["concept_label"].value_counts().rename("n_samples"))

## 3. LLM and SAE loading

In [ ]:
if not SAE_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"SAE checkpoint not found: {SAE_CHECKPOINT_PATH}. "
        "Сначала обучите SAE или укажите существующий checkpoint."
    )

llm = LLM(
    model_name_or_path=MODEL_NAME_OR_PATH, 
    device=DEVICE
)
if llm.tokenizer.pad_token is None:
    llm.tokenizer.pad_token = llm.tokenizer.eos_token

model_device, model_dtype = llm._model_device_dtype()
sae = load_sae_checkpoint(
    checkpoint_path=SAE_CHECKPOINT_PATH,
    device=str(model_device),
    dtype=model_dtype,
)
sae.eval()

named_modules = dict(llm.model.named_modules())
if LAYER_NAME not in named_modules:
    raise ValueError(f"Layer not found: {LAYER_NAME}")

print(f"model_device={model_device}, model_dtype={model_dtype}")
print(f"sae_hidden={sae.in_hidden_state_size}, sae_latent={sae.sparse_hidden_state_size}")

## 4. Token-level  activations extracting

In [ ]:
def module_output_to_hidden_state(output):
    if torch.is_tensor(output):
        return output
    if isinstance(output, (tuple, list)):
        return output[0]
    raise TypeError(f"Unsupported layer output type: {type(output)}")


def extract_layer_activations(
    llm: LLM,
    texts: list[str],
    layer_name: str,
    batch_size: int,
    max_length: int,
) -> tuple[torch.Tensor, torch.Tensor, list[list[str]]]:
    module = dict(llm.model.named_modules())[layer_name]
    all_hidden_states: list[torch.Tensor] = []
    all_attention_masks: list[torch.Tensor] = []
    all_tokens: list[list[str]] = []

    for start in tqdm(range(0, len(texts), batch_size), desc="Extract layer activations"):
        batch_texts = texts[start:start + batch_size]
        inputs = llm.tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=max_length,
        ).to(llm.device)
        saved_hidden_states: list[torch.Tensor] = []

        def hook_fn(module, inputs, output):
            saved_hidden_states.append(module_output_to_hidden_state(output).detach().cpu())

        handle = module.register_forward_hook(hook_fn)
        try:
            with torch.no_grad():
                _ = llm.model(**inputs)
        finally:
            handle.remove()

        if not saved_hidden_states:
            raise RuntimeError(f"Layer did not produce hidden states: {layer_name}")

        hidden_states = saved_hidden_states[-1]
        attention_mask = inputs["attention_mask"].detach().cpu().bool()
        if hidden_states.shape[:2] != attention_mask.shape:
            raise RuntimeError(
                f"Activation shape {tuple(hidden_states.shape)} is incompatible with "
                f"attention mask shape {tuple(attention_mask.shape)}"
            )

        batch_tokens = [
            llm.tokenizer.convert_ids_to_tokens(row)
            for row in inputs["input_ids"].detach().cpu()
        ]
        all_hidden_states.append(hidden_states)
        all_attention_masks.append(attention_mask)
        all_tokens.extend(batch_tokens)

    return torch.cat(all_hidden_states, dim=0), torch.cat(all_attention_masks, dim=0), all_tokens


h, attention_mask, tokens = extract_layer_activations(
    llm=llm,
    texts=texts,
    layer_name=LAYER_NAME,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
)

print(f"h={tuple(h.shape)}, valid_tokens={int(attention_mask.sum().item())}")

## 5. SAE encoding

In [ ]:
def encode_with_sae(
    sae,
    h: torch.Tensor,
    attention_mask: torch.Tensor,
    batch_size: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    device = next(sae.parameters()).device
    z_batches: list[torch.Tensor] = []
    h_hat_batches: list[torch.Tensor] = []

    for start in tqdm(range(0, h.shape[0], batch_size), desc="Encode SAE latents"):
        batch_h = h[start:start + batch_size].to(device)
        with torch.no_grad():
            output = sae(batch_h, return_output=True)
        z_batches.append(output.latent_activation.detach().cpu())
        h_hat_batches.append(output.reconstructed_hidden_state.detach().cpu())

    z = torch.cat(z_batches, dim=0)
    h_hat = torch.cat(h_hat_batches, dim=0)
    z = z.masked_fill(~attention_mask.unsqueeze(-1), 0.0)
    h_hat = h_hat.masked_fill(~attention_mask.unsqueeze(-1), 0.0)
    return z, h_hat


z, h_hat = encode_with_sae(sae=sae, h=h, attention_mask=attention_mask, batch_size=BATCH_SIZE)
print(f"z={tuple(z.shape)}, h_hat={tuple(h_hat.shape)}")

## 6. Reconstruction and sparsity metrics

In [ ]:
valid_h = h[attention_mask].reshape(-1, h.shape[-1]).float()
valid_h_hat = h_hat[attention_mask].reshape(-1, h_hat.shape[-1]).float()
valid_z = z[attention_mask].reshape(-1, z.shape[-1]).float()

reconstruction = sae.reconstruction_metrics(valid_h, valid_h_hat)
sparsity = sae.sparsity_metrics(valid_z, threshold=ACTIVATION_THRESHOLD)
metrics = {
    "mse": float(reconstruction["mse"].item()),
    "nmse": float(reconstruction["nmse"].item()),
    "cosine_similarity": float(reconstruction["cosine_similarity"].item()),
    "l0": float(sparsity["l0"].item()),
    "active_feature_share": float(sparsity["active_feature_share"].item()),
    "hoyer_sparsity": float(sparsity["hoyer_sparsity"].item()),
    "normalized_entropy": float(sparsity["normalized_entropy"].item()),
}
metrics

## 7. Concept-level feature attribution

In [ ]:
feature_attributions = attribute_features_to_concepts(
    latent_activations=z,
    concept_labels=concept_labels,
    top_k=TOP_K_FEATURES_PER_CONCEPT,
    score_method=SCORE_METHOD,
    activation_threshold=ACTIVATION_THRESHOLD,
    token_aggregation=TOKEN_AGGREGATION,
)
feature_concept_scores_df = pd.DataFrame([item.to_dict() for item in feature_attributions])
feature_concept_scores_df = feature_concept_scores_df.sort_values(
    ["concept_label", "score"],
    ascending=[True, False],
).reset_index(drop=True)

feature_concept_scores_path = OUTPUT_DIR / "feature_concept_scores.csv"
feature_concept_scores_df.to_csv(feature_concept_scores_path, index=False)

selected_feature_ids = (
    feature_concept_scores_df.sort_values("score", ascending=False)["feature_index"]
    .drop_duplicates()
    .head(MAX_DASHBOARD_FEATURES)
    .astype(int)
    .tolist()
)

print(f"selected_features={len(selected_feature_ids)}")
display(feature_concept_scores_df.head(20))

## 8. Feature dashboard summary

In [ ]:
feature_summary_rows = []
for feature_id in selected_feature_ids:
    values = valid_z[:, feature_id]
    related = feature_concept_scores_df[feature_concept_scores_df["feature_index"].eq(feature_id)]
    top_concepts = related.sort_values("score", ascending=False).head(5)
    feature_summary_rows.append(
        {
            "feature_index": int(feature_id),
            "mean_activation": float(values.mean().item()),
            "max_activation": float(values.max().item()),
            "activation_density": float((values > ACTIVATION_THRESHOLD).float().mean().item()),
            "top_concepts": json.dumps(
                [
                    {
                        "concept_label": row.concept_label,
                        "score": float(row.score),
                        "mean_inside": float(row.mean_inside),
                        "mean_outside": float(row.mean_outside),
                    }
                    for row in top_concepts.itertuples(index=False)
                ],
                ensure_ascii=False,
            ),
        }
    )

feature_dashboard_df = pd.DataFrame(feature_summary_rows).sort_values(
    ["max_activation", "mean_activation"],
    ascending=False,
).reset_index(drop=True)
feature_dashboard_path = OUTPUT_DIR / "feature_dashboard.csv"
feature_dashboard_df.to_csv(feature_dashboard_path, index=False)

display(feature_dashboard_df.head(20))

## 9. Top activating tokens for selected features

For each pair `(concept_label, feature_index)` we are saving tokens which activate certain feature more than other ones.

In [ ]:
token_rows = []
z_for_tokens = z.clone().float()
z_for_tokens = z_for_tokens.masked_fill(~attention_mask.unsqueeze(-1), float("-inf"))

selected_feature_set = set(selected_feature_ids)
selected_concept_scores_df = feature_concept_scores_df[
    feature_concept_scores_df["feature_index"].isin(selected_feature_set)
]

for concept_label, concept_scores in tqdm(
    selected_concept_scores_df.groupby("concept_label"),
    total=selected_concept_scores_df["concept_label"].nunique(),
    desc="Top tokens",
):
    feature_indices = concept_scores["feature_index"].astype(int).drop_duplicates().tolist()
    score_by_feature = {
        int(row.feature_index): float(row.score)
        for row in concept_scores.itertuples(index=False)
    }
    attributions = top_tokens_for_features(
        latent_activations=z_for_tokens,
        concept_labels=concept_labels,
        feature_indices=feature_indices,
        tokens=tokens,
        top_k=TOP_K_TOKENS_PER_FEATURE,
        concept_label=concept_label,
        activation_transform="raw",
        context_window=CONTEXT_WINDOW,
    )
    for attribution in attributions:
        item = attribution.to_dict()
        if not np.isfinite(item["activation"]):
            continue
        item["focus_concept"] = concept_label
        item["feature_score"] = score_by_feature.get(int(item["feature_index"]), np.nan)
        item["sample_id"] = sample_ids[item["sample_index"]]
        item["text"] = texts[item["sample_index"]]
        token_rows.append(item)

feature_top_tokens_df = pd.DataFrame(token_rows)
feature_top_tokens_path = OUTPUT_DIR / "feature_top_tokens.csv"
feature_top_tokens_df.to_csv(feature_top_tokens_path, index=False)

display(feature_top_tokens_df.head(30))

## 10. Top examples for selected features

In [ ]:
example_rows = []
for feature_id in tqdm(selected_feature_ids, desc="Top examples"):
    examples = top_examples_for_feature(
        latent_activations=z,
        concept_labels=concept_labels,
        feature_index=int(feature_id),
        texts=texts,
        top_k=TOP_K_EXAMPLES_PER_FEATURE,
        token_aggregation="max_abs",
    )
    for example in examples:
        item = example.to_dict()
        item["sample_id"] = sample_ids[item["sample_index"]]
        example_rows.append(item)

feature_top_examples_df = pd.DataFrame(example_rows)
feature_top_examples_path = OUTPUT_DIR / "feature_top_examples.csv"
feature_top_examples_df.to_csv(feature_top_examples_path, index=False)

display(feature_top_examples_df.head(30))

## 11. Per-token top features

In [ ]:
sample_token_rows = []
for sample_index in tqdm(range(z.shape[0]), desc="Per-token features"):
    valid_positions = attention_mask[sample_index].nonzero(as_tuple=False).flatten().tolist()
    for token_position in valid_positions:
        token_latent = z[sample_index, token_position].float()
        values, indices = torch.topk(
            token_latent,
            k=min(TOP_K_FEATURES_PER_TOKEN, token_latent.numel()),
        )
        for rank, (feature_id_tensor, activation_tensor) in enumerate(zip(indices, values), start=1):
            sample_token_rows.append(
                {
                    "sample_id": sample_ids[sample_index],
                    "sample_index": int(sample_index),
                    "concept_label": concept_labels[sample_index],
                    "token_position": int(token_position),
                    "token_text": tokens[sample_index][token_position],
                    "feature_rank": rank,
                    "feature_index": int(feature_id_tensor.item()),
                    "activation": float(activation_tensor.item()),
                    "text": texts[sample_index],
                }
            )

sample_token_attributions_df = pd.DataFrame(sample_token_rows)
sample_token_attributions_path = OUTPUT_DIR / "sample_token_attributions.csv"
sample_token_attributions_df.to_csv(sample_token_attributions_path, index=False)

display(sample_token_attributions_df.head(30))

## 12. Metadata and result files

In [ ]:
metadata = {
    **config,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "n_samples": int(len(dataset_df)),
    "n_valid_tokens": int(attention_mask.sum().item()),
    "hidden_size": int(h.shape[-1]),
    "latent_size": int(z.shape[-1]),
    "metrics": metrics,
    "files": {
        "feature_concept_scores": str(feature_concept_scores_path),
        "feature_dashboard": str(feature_dashboard_path),
        "feature_top_tokens": str(feature_top_tokens_path),
        "feature_top_examples": str(feature_top_examples_path),
        "sample_token_attributions": str(sample_token_attributions_path),
    },
    "method_note": (
        "Dashboard feature labels and rankings are activation-based candidates. "
        "Causal claims require baseline/intervention checks."
    ),
}
metadata_path = OUTPUT_DIR / "dashboard_metadata.json"
with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Saved dashboard artifacts:")
for name, path in metadata["files"].items():
    print(f"- {name}: {path}")
print(f"- metadata: {metadata_path}")